# BurstGPT 3: train, validate, then compare frozen vs adapting copies
This is a **self-contained notebook**. Supply only `BurstGPT_3.csv` through Google Drive or upload. You do **not** need a results ZIP, saved checkpoint, or another notebook run.

Workflow: setup → prepare selected timestamp ranges → offline DQN training → validation/checkpoint selection → copy selected weights → frozen and online-adapting replays of identical traffic → add the three rule-based policies on the same continuous traffic → unified charts and report.

The offline training algorithm, reward, validation selection, and default 24-episode budget are unchanged from the original notebook. Both comparison copies begin with the **same validation-selected checkpoint**. The frozen copy makes no updates. The adapting copy learns only from feedback already delivered by the simulator, never future outcomes; its replay buffer starts empty. Default exploration is zero.

This is a separate **online-learning evaluation**, not a frozen held-out score. Do not tune settings on these outcomes and call them an untouched test. Compare the paired continuous replays here, not the earlier notebook's episodic frozen scores.

CPU and GPU runtimes both work. `DEVICE="auto"` selects CUDA minibatch learning if available, otherwise CPU. Select a Colab GPU runtime to try it; GPU service is still simulated on CPU. The complete test ranges contain over 1.8 million requests and continuous replay retains lifecycle events, so use a high-RAM runtime for a full run. For an initial workflow check, set `MAX_SECONDS_PER_GROUP=1800` to use a 30-minute prefix of each group. The default `None` uses the complete selected ranges.

The initial offline budget is exploratory: 24 sampled training episodes, not a pass through every training window. For final results use multiple training seeds and a larger justified budget.

Only neural-network minibatch learning can use CUDA; the simulator, baseline evaluation, and single-request inference remain on CPU. This small network and frequent transfers can make GPU mode slower. Compare measured runtime with `DEVICE="cpu"`; no end-to-end speedup is promised. Auto chooses by availability, not a speed benchmark. CUDA and CPU numerics need not produce identical training trajectories. Apple MPS is not selected; it falls back to CPU.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "burstgpt_3"  # Change to main after this branch is merged.
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    REPO = Path("/content/llm-request-triage")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "status", "--short"], check=True)
        print("Using the existing checkout. Fetch/switch explicitly if you need another revision.")
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent

required = ["experiments/online_adaptation.py", "configs/burstgpt3_ranges.json",
            "scripts/prepare_burstgpt3.py", "src/llm_triage/workload/time_ranges.py"]
assert all((REPO / name).is_file() for name in required), "Checkout lacks BurstGPT 3 changes; use the pushed burstgpt_3 branch."
os.chdir(REPO)
assert sys.version_info >= (3, 11), "Use Python 3.11 or newer."
print("Repository:", REPO)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
extras = ".[dev,rl,analysis,gpu]" if IS_COLAB else ".[dev,rl,analysis]"
subprocess.run([sys.executable, "-m", "pip", "install", "-e", extras, "pandas>=2.0"], check=True)
sys.path.insert(0, str(REPO / "src"))

import base64
import csv
from datetime import datetime, timezone
import hashlib
import html
import importlib.metadata
import json
import shutil
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

from llm_triage.workload.burstgpt import sha256


## Run settings
Only the CSV is an input. `CSV_MODE="drive"` mounts Drive and copies the CSV to Colab's local disk; alternatively choose upload.

Use a blank `RUN_NAME` for a fresh complete training-and-comparison run. Reuse a name only for the exact same settings and completed artifacts. Offline training supports checked resume; interrupted online comparisons require a fresh run name.

`SEEDS` controls offline training. `ADAPTATION_SEED` chooses which trained checkpoint starts **both** comparison copies. Online learning rate and exploration are separate experimental settings; they do not alter offline training.

`DEVICE` changes the learner execution device, not simulated GPU hardware. `auto` uses CUDA if PyTorch and CUDA are available; `cpu` forces the original NumPy learner; `cuda` requires a working CUDA runtime. The Colab installation includes the optional PyTorch dependency. For local GPU use, install `.[gpu]` as well.


In [ ]:
CSV_MODE = "drive"  # @param ["drive", "upload"]
DRIVE_CSV_PATH = "/content/drive/MyDrive/BurstGPT_3.csv"  # @param {type:"string"}
LOCAL_CSV_PATH = str(REPO / "BurstGPT_3.csv")  # Used when running locally.
RUN_NAME = ""  # @param {type:"string"}
SEEDS = [7]  # For final experiments consider [7, 17, 27].
EPISODES = 24  # @param {type:"integer"}
VALIDATE_EVERY = 12  # @param {type:"integer"}
EPISODE_SECONDS = 300  # @param {type:"integer"}
TRAIN_TARGET_RPS = [None]  # None preserves the original traffic timing.
RUN_REPOSITORY_TESTS = True  # @param {type:"boolean"}
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
from llm_triage.rl.devices import resolve_device
print("Requested learner:", DEVICE, "| Selected minibatch device:", resolve_device(DEVICE))
print("Simulator and single-request inference run on CPU in all modes.")

ADAPTATION_SEED = 7  # Must be in SEEDS.
ONLINE_LEARNING_RATE = 0.0001
EPSILON = 0.0  # Only the adapting copy explores, if nonzero.
BATCH_SIZE = 64
WARMUP = 256
UPDATE_INTERVAL = 4
REPLAY_CAPACITY = 50000
MAX_SECONDS_PER_GROUP = None  # Full ranges. Use 1800 for a short check.
GROUPS = None  # Both groups, in chronological order.
RUN_ONLINE_EXPERIMENT = True  # False allows stopping after offline validation.
RUN_RULE_BASELINES = True  # Add FCFS, length-aware and static SLO-aware on the same continuous traffic.
SEED = ADAPTATION_SEED
assert SEED in SEEDS, "Include ADAPTATION_SEED in the offline training SEEDS."

run_name = RUN_NAME or datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
assert Path(run_name).name == run_name and run_name not in {".", ".."}, "Use a simple folder name."
RUN_ROOT = REPO / "results" / "runs" / "colab_burstgpt3_online" / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES = RUN_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
DATA_DIR = RUN_ROOT / "data"
ANALYSIS_DIR = RUN_ROOT / "traffic_analysis"
TRAIN_OUTPUT = RUN_ROOT / "training_run"
COMMAND_LOG = RUN_ROOT / "notebook_commands.log"
RANGE_CONFIG_PATH = RUN_ROOT / "ranges.json"
TRAIN_CONFIG_PATH = RUN_ROOT / "training_config.json"
generated_figures = set()
print("Run folder:", RUN_ROOT)

def run_command(args):
    command = [str(value) for value in args]
    print("Running:", " ".join(command), flush=True)
    with COMMAND_LOG.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def save_figure(fig, name):
    path = FIGURES / (name + ".png")
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    generated_figures.add(path.name)
    display(Image(filename=str(path)))
    return path

def save_config(path, value):
    if path.exists():
        assert json.loads(path.read_text()) == value, "Existing run config differs. Use a new RUN_NAME."
    else:
        path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


In [ ]:
if IS_COLAB:
    SOURCE_CSV = REPO / "data" / "raw" / "BurstGPT_3.csv"
    SOURCE_CSV.parent.mkdir(parents=True, exist_ok=True)
    if not SOURCE_CSV.exists():
        if CSV_MODE == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            drive_source = Path(DRIVE_CSV_PATH)
            assert drive_source.is_file(), "Set DRIVE_CSV_PATH to your BurstGPT_3.csv."
            shutil.copyfile(drive_source, SOURCE_CSV)
        elif CSV_MODE == "upload":
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if Path(name).suffix.lower() == ".csv"]
            assert len(csv_files) == 1, "Upload exactly one BurstGPT CSV."
            SOURCE_CSV.write_bytes(uploaded[csv_files[0]])
            del uploaded
        else:
            raise ValueError("CSV_MODE must be drive or upload")
else:
    SOURCE_CSV = Path(LOCAL_CSV_PATH).expanduser().resolve()

assert SOURCE_CSV.is_file(), "BurstGPT_3.csv was not found."
print("CSV:", SOURCE_CSV, "Bytes:", SOURCE_CSV.stat().st_size)
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q"])


## Inspect and record the dataset
The ranges below use **original CSV timestamps in seconds**, with inclusive starts and exclusive ends. The red-box boundaries were estimated from the annotated plot and aligned to five-minute bins.

To change the research protocol, edit `range_config["ranges"]` here before running preparation. The prepared manifest records empty episodes and source/config checksums. The analysis's automatic 60/20/20 regime labels do not control this explicit-range dataset.

Never adjust the test ranges or checkpoint based on test scores. No-record intervals can mean inactivity or collection gaps; they are not proof of actual idle traffic.


In [ ]:
range_config = json.loads((REPO / "configs/burstgpt3_ranges.json").read_text())
range_config["episode_seconds"] = EPISODE_SECONDS
# Optional: edit range_config["ranges"] here before preparation.
assert sha256(SOURCE_CSV) == range_config["source_sha256"], "CSV does not match the selected BurstGPT 3 source."
save_config(RANGE_CONFIG_PATH, range_config)
display(pd.DataFrame(range_config["ranges"]))

if not (ANALYSIS_DIR / "summary.json").exists():
    run_command([sys.executable, "scripts/analyze_burstgpt_traffic.py", "--source", SOURCE_CSV,
                 "--output-dir", ANALYSIS_DIR, "--bin-seconds", 300])
analysis = json.loads((ANALYSIS_DIR / "summary.json").read_text())
assert analysis["source"]["sha256"] == range_config["source_sha256"]

if not (DATA_DIR / "processed" / "manifest.json").exists():
    run_command([sys.executable, "scripts/prepare_burstgpt3.py", "--source", SOURCE_CSV,
                 "--analysis-dir", ANALYSIS_DIR, "--ranges-config", RANGE_CONFIG_PATH, "--data-dir", DATA_DIR])
manifest = json.loads((DATA_DIR / "processed" / "manifest.json").read_text())
assert manifest["range_config"] == range_config, "Prepared ranges differ. Use a fresh run."
assert manifest["source_sha256"] == sha256(SOURCE_CSV)
assert manifest["selected_sha256"] == sha256(DATA_DIR / "processed" / "burstgpt_windows.csv")

episode_table = pd.DataFrame(manifest["windows"] + manifest.get("empty_windows", []))
episode_table["duration_s"] = episode_table["end_timestamp_s"] - episode_table["start_timestamp_s"]
episode_table["nonempty"] = episode_table["requests"] > 0
dataset_summary = episode_table.groupby(["split", "group"], sort=False).agg(
    requests=("requests", "sum"), episodes=("window", "size"),
    nonempty_episodes=("nonempty", "sum"), duration_s=("duration_s", "sum")).reset_index()
dataset_summary["no_record_episode_fraction"] = 1 - dataset_summary["nonempty_episodes"] / dataset_summary["episodes"]
dataset_summary["mean_rps_including_gaps"] = dataset_summary["requests"] / dataset_summary["duration_s"]
display(dataset_summary)
dataset_summary.to_csv(RUN_ROOT / "dataset_summary.csv", index=False)

traffic = pd.read_csv(ANALYSIS_DIR / "time_bins.csv")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(traffic["start_timestamp_s"], traffic["mean_rps"], linewidth=0.6, color="0.5")
colors = {"train": "#2166ac", "validation": "#4daf4a", "test": "#d95f02"}
for item in range_config["ranges"]:
    ax.axvspan(item["start_timestamp_s"], item["end_timestamp_s"], alpha=0.18,
               color=colors[item["split"]], label=item["group"])
ax.ticklabel_format(axis="x", style="plain", useOffset=False)
ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Five-minute requests/s",
       title="Selected traffic ranges — natural traffic mix retained")
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, "selected_traffic_ranges")


## Train and select using validation
The pipeline trains DQN, chooses checkpoints by **mean validation episode SLO attainment**, reloads them, and compares them with FCFS, length-aware, and static SLO-aware policies. Test periods are reserved.

A training episode is sampled with replacement; the episode budget is not an epoch count. Sparse episodes and repeated deferrals change the number of learning steps. With larger datasets, inspect coverage and learning curves rather than assuming 24 episodes is sufficient.

The pipeline log prints its training-log path. Detailed checkpoint progress is written there while the subprocess runs. An existing run is resumed only with unchanged settings, code, and runtime.


In [ ]:
train_config = json.loads((REPO / "configs/burstgpt3_initial.json").read_text())
train_config.update(seeds=SEEDS, episodes=EPISODES, validate_every=VALIDATE_EVERY, target_rps=TRAIN_TARGET_RPS)
train_config.setdefault("agent", {})["device"] = DEVICE
save_config(TRAIN_CONFIG_PATH, train_config)
display(train_config)
command = [sys.executable, "-m", "experiments.run_pipeline", "--config", TRAIN_CONFIG_PATH,
           "--data-dir", DATA_DIR, "--output", TRAIN_OUTPUT]
if (TRAIN_OUTPUT / "pipeline.json").exists():
    command.append("--resume")
run_command(command)
pipeline_state = json.loads((TRAIN_OUTPUT / "pipeline.json").read_text())
assert pipeline_state["status"] == "completed"
TRAINING_DIR = TRAIN_OUTPUT / "training"
training_record = json.loads((TRAINING_DIR / "run.json").read_text())
assert training_record["test_evaluated"] is False
checkpoint_hashes = {f"dqn_seed{item['seed']}.npz": item["sha256"] for item in training_record["checkpoints"]}
for filename, expected in checkpoint_hashes.items():
    assert sha256(TRAINING_DIR / filename) == expected
display(pd.DataFrame(training_record["checkpoints"]))


In [ ]:
history = pd.read_csv(TRAINING_DIR / "training.csv")
selection = pd.read_csv(TRAINING_DIR / "checkpoints.csv")
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for seed in history["seed"].unique():
    h = history[history["seed"] == seed]
    s = selection[selection["seed"] == seed]
    axes[0, 0].plot(s["episode"], s["validation_attainment"], marker="o", label=f"seed {seed}")
    axes[0, 1].plot(h["episode"], h["train_attainment"], label=f"seed {seed}")
    axes[1, 0].plot(h["episode"], h["reward"], label=f"seed {seed}")
    axes[1, 1].plot(h["episode"], h["loss"], label=f"seed {seed}")
for item in training_record["checkpoints"]:
    axes[0, 0].scatter(item["episode"], item["validation_attainment"], marker="*", s=150, color="black")
axes[0, 0].set_title("Validation attainment; stars mark selected checkpoints")
axes[0, 1].set_title("Training attainment (exploration enabled)")
axes[1, 0].set_title("Episode return (affected by request count)")
axes[1, 1].set_title("Mean training loss")
for ax in axes.flat:
    ax.set_xlabel("Training episode")
    ax.grid(alpha=0.2)
    ax.legend()
for ax in axes[0]:
    ax.yaxis.set_major_formatter(PercentFormatter(1))
save_figure(fig, "learning_curves")
coverage = history.groupby("seed").agg(sampled_episodes=("episode", "size"), unique_windows=("window", "nunique"))
display(coverage)


## Validation comparison
The pipeline's `summary.csv` averages episode scores. Since episodes have different request counts, the chart below also computes **request-weighted** attainment and drop rate. Keep these definitions separate when reporting results.


In [ ]:
validation_runs = pd.read_csv(TRAIN_OUTPUT / "validation.csv")
validation_runs["dropped_estimate"] = validation_runs["drop_rate"] * validation_runs["arrivals"]
validation_summary = validation_runs.groupby("policy", sort=False).agg(
    requests=("arrivals", "sum"), slo_met=("slo_met", "sum"),
    dropped=("dropped_estimate", "sum"), mean_episode_attainment=("slo_attainment", "mean")).reset_index()
validation_summary["overall_slo_attainment"] = validation_summary["slo_met"] / validation_summary["requests"]
validation_summary["overall_drop_rate"] = validation_summary["dropped"] / validation_summary["requests"]
display(validation_summary)
validation_summary.to_csv(RUN_ROOT / "validation_weighted_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, title in zip(axes, ["overall_slo_attainment", "overall_drop_rate"],
                             ["Validation: request-weighted SLO attainment", "Validation: request-weighted drop rate"]):
    ax.bar(validation_summary["policy"], validation_summary[metric])
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(axis="x", rotation=25)
    ax.set_title(title)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.2)
save_figure(fig, "validation_comparison")


## Causal replay protocol
At simulated time `t`, the copy chooses an action from its current sanitized observation. The simulator advances to the next decision and delivers only the intervening rewards/outcomes. Only then may the adapting copy add that transition to replay and update. Its next decision occurs at or after delivery time. Outcomes of requests still in flight are not available for learning. Replay may resample **past transitions**, but never future ones.

Unlike the earlier frozen notebook, this comparison uses **one continuous queue** over the selected ranges, including idle gaps. Queues and learning state do not reset every five minutes. Five-minute windows are reporting bins only. The existing time-progress observation is normalized by the total replay horizon for **both copies**, which differs from offline episodic training; compare these paired runs to each other, not directly to the earlier episodic scores.

Traffic outside the selected ranges is omitted and treated as idle, not reconstructed. GPU behavior is simulated; learner computation does not add simulated GPU latency. Training uses existing delayed environment rewards (including shaping), not future true output lengths. The final terminal bonus is delivered only at stream termination and cannot improve earlier decisions.


In [ ]:
ONLINE_OUTPUT = RUN_ROOT / "online_comparison"
online_settings = dict(seed=SEED, groups=GROUPS, epsilon=EPSILON, learning_rate=ONLINE_LEARNING_RATE,
                       batch_size=BATCH_SIZE, warmup=WARMUP, update_interval=UPDATE_INTERVAL,
                       capacity=REPLAY_CAPACITY, max_seconds_per_group=MAX_SECONDS_PER_GROUP, device=DEVICE)
display(pd.DataFrame([online_settings]))
if not RUN_ONLINE_EXPERIMENT:
    print("Not run. Set RUN_ONLINE_EXPERIMENT=True above to start the comparison.")
else:
    if ONLINE_OUTPUT.exists():
        state = json.loads((ONLINE_OUTPUT / "run.json").read_text())
        assert state["status"] == "completed", "Incomplete online runs cannot resume. Use a fresh RUN_NAME."
        assert state["settings"] == online_settings, "Settings changed. Use a fresh RUN_NAME."
        assert state["source_checkpoint_sha256"] == sha256(TRAINING_DIR / f"dqn_seed{SEED}.npz")
        assert state["manifest_sha256"] == sha256(DATA_DIR / "processed/manifest.json")
        for policy, digest in state["final_checkpoints"].items():
            assert sha256(ONLINE_OUTPUT / f"{policy}_final.npz") == digest
        print("Reusing completed comparison; not replaying or training a second time.")
    else:
        command = [sys.executable, "-m", "experiments.online_adaptation",
                   "--training-dir", TRAINING_DIR, "--data-dir", DATA_DIR, "--output", ONLINE_OUTPUT,
                   "--seed", SEED, "--epsilon", EPSILON, "--learning-rate", ONLINE_LEARNING_RATE,
                   "--batch-size", BATCH_SIZE, "--warmup", WARMUP, "--update-interval", UPDATE_INTERVAL,
                   "--capacity", REPLAY_CAPACITY, "--device", DEVICE]
        if GROUPS is not None:
            command += ["--groups", *GROUPS]
        if MAX_SECONDS_PER_GROUP is not None:
            command += ["--max-seconds-per-group", MAX_SECONDS_PER_GROUP]
        run_command(command)


In [ ]:
assert RUN_ONLINE_EXPERIMENT and (ONLINE_OUTPUT / "summary.csv").exists(), "Run the online comparison first."
summary = pd.read_csv(ONLINE_OUTPUT / "summary.csv")
timeline = pd.read_csv(ONLINE_OUTPUT / "timeline.csv")
learning = pd.read_csv(ONLINE_OUTPUT / "learning_summary.csv")
display(summary[["group", "policy", "arrivals", "slo_attainment", "drop_rate", "window_goodput_rps"]])
display(learning)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ["slo_attainment", "drop_rate", "window_goodput_rps"],
                            ["SLO attainment", "Drop rate", "Goodput (req/s)"]):
    summary.pivot(index="group", columns="policy", values=metric).plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    if metric != "window_goodput_rps":
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.set_ylim(0, 1.05 if metric == "slo_attainment" else max(0.05, float(summary[metric].max()) * 1.1))
save_figure(fig, "online_comparison")

group_names = list(timeline["group"].drop_duplicates())
fig, axes = plt.subplots(len(group_names), 1, figsize=(13, 3.5 * len(group_names)), squeeze=False)
for ax, group in zip(axes[:, 0], group_names):
    for policy, rows in timeline[timeline["group"] == group].groupby("policy"):
        rows = rows.sort_values("source_start_s")
        # Leave no-record bins missing rather than misrepresenting them as 0% SLO.
        score = rows["slo_attainment"].where(rows["arrivals"] > 0)
        ax.plot(rows["source_start_s"], score, label=policy)
    ax.set(title=f"{group}: arrival-cohort SLO attainment", xlabel="Original CSV timestamp (seconds)", ylim=(0, 1.05))
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend()
save_figure(fig, "online_timeline")

audit = pd.read_csv(ONLINE_OUTPUT / "adapting_updates.csv")
if not audit.empty:
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(audit["source_timestamp_s"], audit["updates"])
    ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Cumulative updates", title="Learning only as simulated time advances")
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    save_figure(fig, "online_updates")
else:
    print("No updates: selected traffic produced fewer steps than warmup. Increase the prefix length.")
delta = summary.pivot(index="group", columns="policy", values="slo_attainment")
delta["adapting_minus_frozen_percentage_points"] = 100 * (delta["adapting"] - delta["frozen"])
display(delta)
delta.to_csv(RUN_ROOT / "attainment_delta.csv")


## Interpret the result
Compare request-weighted SLO attainment, drops, and goodput **and** how performance changes over time. Learning can improve or degrade performance; neither is guaranteed. Nonzero exploration adds an exploration cost, so report it explicitly. One seed and a short prefix are a workflow check, not evidence of general improvement.

Timeline SLO uses eventual outcomes of each arrival cohort for **offline reporting only**; it is not information fed to earlier decisions. Goodput is cohort-local: on-time completions of a group's arrivals within that group's interval, not all completions flowing across its boundary. Requests arriving in empty bins are zero; no-record periods may be collection gaps. We retain delayed completions beyond the final arrival interval when computing completion SLO.


In [ ]:
provenance = dict(
    experiment="Online adaptation comparison (not frozen held-out evaluation)",
    git_revision=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    python=sys.version, versions={name: importlib.metadata.version(name) for name in ("numpy", "gymnasium", "pandas", "matplotlib")},
    local_csv_path=str(SOURCE_CSV), source_sha256=sha256(SOURCE_CSV),
    source_checkpoints=checkpoint_hashes, training_config=train_config,
    settings=online_settings,
    code_sha256={p.relative_to(REPO).as_posix(): sha256(p) for p in
                 [REPO / "pyproject.toml", *sorted((REPO / "src").rglob("*.py")),
                  *sorted((REPO / "experiments").glob("*.py")), *sorted((REPO / "scripts").glob("*.py"))]})
save_config(RUN_ROOT / "online_provenance.json", provenance)
parts = ["<!doctype html><html><head><meta charset='utf-8'><title>Online adaptation</title>",
         "<style>body{font:16px system-ui;max-width:1200px;margin:30px auto;padding:20px}img{max-width:100%}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px}pre{white-space:pre-wrap}</style></head><body>",
         "<h1>Frozen vs online-adapting DQN</h1><p>Online experiment, not a frozen held-out test. Identical starting weights and native traffic. Continuous queues; past-only replay; no learner wall-time in simulated GPU service.</p>",
         "<h2>Offline training settings</h2><pre>" + html.escape(json.dumps(train_config, indent=2)) + "</pre>",
         "<h2>Dataset</h2>", dataset_summary.to_html(index=False),
         "<h2>Selected offline checkpoints</h2>", pd.DataFrame(training_record["checkpoints"]).to_html(index=False),
         "<h2>Validation comparison</h2>", validation_summary.to_html(index=False),
         "<h2>Adaptation settings</h2><pre>" + html.escape(json.dumps(online_settings, indent=2)) + "</pre>",
         "<h2>Summary</h2>", summary.to_html(index=False), "<h2>Learning</h2>", learning.to_html(index=False),
         "<p>Timeline scores use eventual arrival-cohort outcomes for reporting only, never for earlier decisions. Goodput counts cohort-local completions within each interval. See selected_ranges in run.json for the exact comparison intervals (full ranges by default).</p>"]
for name in sorted(generated_figures):
    encoded = base64.b64encode((FIGURES / name).read_bytes()).decode("ascii")
    parts.append("<img alt='" + html.escape(name) + "' src='data:image/png;base64," + encoded + "'>")
parts += ["<h2>Provenance</h2><pre>", html.escape(json.dumps(provenance, indent=2)), "</pre></body></html>"]
REPORT = RUN_ROOT / "online_report.html"
REPORT.write_text("\n".join(parts), encoding="utf-8")
ARCHIVE = RUN_ROOT / (run_name + "_online_results.zip")
# Explicit allowlist: no raw CSV or prepared request data.
members = [REPORT, RUN_ROOT / "online_provenance.json", RUN_ROOT / "attainment_delta.csv", RANGE_CONFIG_PATH,
           TRAIN_CONFIG_PATH, RUN_ROOT / "dataset_summary.csv", RUN_ROOT / "validation_weighted_summary.csv",
           TRAIN_OUTPUT / "pipeline.json", TRAIN_OUTPUT / "validation.csv",
           TRAINING_DIR / "training.csv", TRAINING_DIR / "checkpoints.csv",
           TRAIN_OUTPUT / pipeline_state["training_log"]]
members += [ONLINE_OUTPUT / name for name in ("summary.csv", "timeline.csv", "learning_summary.csv", "run.json",
            "frozen_updates.csv", "adapting_updates.csv", "frozen_final.npz", "adapting_final.npz")]
members += [TRAINING_DIR / "run.json", *[TRAINING_DIR / name for name in checkpoint_hashes],
            DATA_DIR / "processed/manifest.json", COMMAND_LOG]
members += [FIGURES / name for name in sorted(generated_figures)]
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for p in members:
        bundle.write(p, p.relative_to(RUN_ROOT).as_posix())
display(FileLink(str(REPORT)))
display(FileLink(str(ARCHIVE)))
if IS_COLAB and not RUN_RULE_BASELINES:
    from google.colab import files
    files.download(str(ARCHIVE))
print("Also save a notebook copy with outputs in Drive to share the executed analysis.")


## Unified continuous test comparison: add the three rule-based baselines
Run this after frozen/adapting replay completes. It does **not** train or replay either DQN copy. It reads the exact saved ranges and settings, runs FCFS, length-aware and static SLO-aware on independent continuous simulations, and combines all five policies into `unified_report.html`, CSVs, charts, and `unified_results.zip`. Completed baselines are cached individually; rerunning this cell skips them if inputs and outputs are unchanged.

These are additional full-trace simulations, so they can take time. Keep the original runtime/output folders and code revision; the HTML report alone is not enough. To patch an already completed Colab notebook, copy the complete contents of `notebooks/colab_add_continuous_baselines.py` into a new final cell and run **only that cell**. No Git update or new package is needed for that paste-in patch.


In [ ]:
# Paste this entire file into ONE NEW CELL after the existing online comparison
# and export cells. It needs the original Colab runtime/output folders, not just
# the downloaded HTML. No training or frozen/adapting DQN replay is performed.
from pathlib import Path
import base64
import html
import json
import math
import time
import zipfile

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image, FileLink

from llm_triage.metrics import evaluate_events
from llm_triage.metrics.summary import summarize_records
from llm_triage.scheduling import baseline_schedulers
from llm_triage.simulation import HardwareConfig, ServiceProfile, simulate
from llm_triage.workload.burstgpt import load_windows, sha256
from llm_triage.workload.transforms import replay_window


def add_continuous_baselines(online_dir, training_dir, data_dir, destination):
    """Add missing policies using the completed run's exact continuous replay."""
    completed = json.loads((online_dir / "run.json").read_text())
    assert completed["status"] == "completed", "Finish frozen/adapting replay first."
    assert completed["protocol"] == "online_prequential_continuous_v1"
    training_path = training_dir / "run.json"
    training = json.loads(training_path.read_text())
    assert sha256(training_path) == completed["source_training_record_sha256"]
    manifest_path = data_dir / "processed/manifest.json"
    manifest = json.loads(manifest_path.read_text())
    trace_path = data_dir / "processed/burstgpt_windows.csv"
    assert sha256(manifest_path) == completed["manifest_sha256"]
    assert sha256(trace_path) == completed["trace_sha256"] == manifest["selected_sha256"]
    assert manifest["source_sha256"] == completed["source_sha256"]
    checkpoint = training_dir / f"dqn_seed{completed['settings']['seed']}.npz"
    assert sha256(checkpoint) == completed["source_checkpoint_sha256"]
    old_summary = pd.read_csv(online_dir / "summary.csv")
    old_timeline = pd.read_csv(online_dir / "timeline.csv")
    assert set(old_summary["policy"]) == {"frozen", "adapting"}

    # Verify relevant installed code against the original exported provenance,
    # when available. Do not pull/switch simulator versions between comparisons.
    repo = Path.cwd()
    source_paths = [p for folder in ("simulation", "scheduling", "metrics", "workload")
                    for p in (repo / "src/llm_triage" / folder).rglob("*.py")]
    assert source_paths, "Run from the repository root, as in the setup cell."
    code = {p.relative_to(repo).as_posix(): sha256(p) for p in sorted(source_paths)}
    original_provenance = online_dir.parent / "online_provenance.json"
    if original_provenance.exists():
        original_code = json.loads(original_provenance.read_text())["code_sha256"]
        assert all(original_code.get(k) == v for k, v in code.items()), "Simulator code changed; restore the original checkout."
    identity = dict(protocol="continuous_baseline_addon_v1", online_run_sha256=sha256(online_dir / "run.json"),
                    summary_sha256=sha256(online_dir / "summary.csv"),
                    timeline_sha256=sha256(online_dir / "timeline.csv"), code_sha256=code)
    destination.mkdir(parents=True, exist_ok=True)
    ranges = completed["selected_ranges"]  # Includes any prefix/group selection actually used.
    origin, end = ranges[0]["start_timestamp_s"], ranges[-1]["end_timestamp_s"]
    bin_seconds = manifest["range_config"]["episode_seconds"]
    replay, labels = None, None
    summary_frames, timeline_frames = [old_summary], [old_timeline]

    class Progress:
        def __init__(self, name, scheduler):
            self.name, self.scheduler, self.decisions = name, scheduler, 0

        def __getattr__(self, name):
            return getattr(self.scheduler, name)  # Preserve each baseline's priority rule.

        def decide(self, state):
            self.decisions += 1
            if self.decisions % 100000 == 0:
                print(f"{self.name}: decisions={self.decisions}, source_time={origin + state.now_s:.1f}", flush=True)
            return self.scheduler.decide(state)

    for name, scheduler in baseline_schedulers().items():
        folder = destination / name
        record_path = folder / "run.json"
        saved = json.loads(record_path.read_text()) if record_path.exists() else None
        if saved is not None:
            assert saved["identity"] == identity, "Cached inputs differ; choose a new destination folder."
        if saved is not None and saved["status"] == "completed":
            for filename, digest in saved["files"].items():
                assert sha256(folder / filename) == digest, "Cached baseline output changed."
            print("Reusing completed baseline:", name)
        else:
            # Load requests only if a baseline still needs to run. Preserve the
            # exact sorting, source IDs, bounds, idle gaps and native time scale.
            if replay is None:
                windows = load_windows(trace_path)
                rows = sorted((r for (split, _), values in windows.items() if split == "test"
                               for r in values if any(g["start_timestamp_s"] <= r.timestamp_s < g["end_timestamp_s"]
                                                      for g in ranges)), key=lambda r: (r.timestamp_s, r.source_row))
                replay = replay_window(rows, slo_s=training["config"]["slo_s"], bounds=(origin, end))
                labels = {}
                for row in rows:
                    group = next(g for g in ranges if g["start_timestamp_s"] <= row.timestamp_s < g["end_timestamp_s"])
                    labels[str(row.source_row)] = (group["group"], int((row.timestamp_s-group["start_timestamp_s"]) // bin_seconds))
                del windows, rows
                for group in ranges:
                    expected = sum(label[0] == group["group"] for label in labels.values())
                    paired = old_summary[old_summary["group"] == group["group"]]
                    assert len(paired) == 2 and (paired["arrivals"] == expected).all(), "DQN cohort differs."
            folder.mkdir(exist_ok=True)
            record_path.write_text(json.dumps(dict(identity=identity, status="running"), indent=2))
            print("Running continuous baseline:", name, flush=True)
            started = time.monotonic()
            callback = Progress(name, scheduler)
            simulation = simulate(replay.requests, callback,
                                  HardwareConfig(**training["config"]["hardware"]),
                                  ServiceProfile(**training["service_profile"]),
                                  estimated_output_tokens=training["estimated_output_tokens"],
                                  record_iterations=False, **training["config"]["decisions"])
            result = evaluate_events(simulation.events, measurement_window_s=replay.horizon_s)
            assert {r.request_id for r in result.request_records} == set(labels), "Unpaired baseline cohort."
            by_group, by_bin = {}, {}
            for row in result.request_records:
                group, index = labels[row.request_id]
                by_group.setdefault(group, []).append(row)
                by_bin.setdefault((group, index), []).append(row)
            scores, timeline = [], []
            for group in ranges:
                label = group["group"]
                start, stop = group["start_timestamp_s"], group["end_timestamp_s"]
                score = summarize_records(by_group.get(label, []), measurement_window_s=stop-start, window_start_s=start-origin)
                scores.append(dict(policy=name, group=label, **score))
                for index in range(math.ceil((stop-start) / bin_seconds)):
                    a, b = start + index * bin_seconds, min(stop, start + (index+1) * bin_seconds)
                    score = summarize_records(by_bin.get((label, index), []), measurement_window_s=b-a, window_start_s=a-origin)
                    timeline.append(dict(policy=name, group=label, source_start_s=a, source_end_s=b, **score))
            pd.DataFrame(scores).to_csv(folder / "summary.csv", index=False)
            pd.DataFrame(timeline).to_csv(folder / "timeline.csv", index=False)
            record_path.write_text(json.dumps(dict(identity=identity, status="completed", policy=name,
                decisions=callback.decisions, wall_seconds=time.monotonic()-started,
                selected_ranges=ranges, training_config=training["config"],
                service_profile=training["service_profile"],
                files={f: sha256(folder / f) for f in ("summary.csv", "timeline.csv")}), indent=2))
            print(name, "completed", flush=True)
            del simulation, result, by_group, by_bin
        summary_frames.append(pd.read_csv(folder / "summary.csv"))
        timeline_frames.append(pd.read_csv(folder / "timeline.csv"))

    unified_summary = pd.concat(summary_frames, ignore_index=True)
    unified_timeline = pd.concat(timeline_frames, ignore_index=True)
    order = ["fcfs", "length_aware", "static_slo_aware", "frozen", "adapting"]
    expected = {(g["group"], p) for g in ranges for p in order}
    assert set(zip(unified_summary["group"], unified_summary["policy"])) == expected
    assert not unified_summary.duplicated(["group", "policy"]).any()
    unified_summary = unified_summary.sort_values(["group", "policy"]).reset_index(drop=True)
    unified_summary.to_csv(destination / "unified_summary.csv", index=False)
    unified_timeline.to_csv(destination / "unified_timeline.csv", index=False)
    (destination / "provenance.json").write_text(json.dumps(dict(identity=identity,
        original_online_run=completed, original_training_record=training,
        simulator_code_verified_against_export=original_provenance.exists(),
        notes="Independent continuous replays; same native traffic, bounds, gaps, capacity, SLO and train-only length estimate. No DQN replay or training repeated."), indent=2))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, metric, title in zip(axes, ["slo_attainment", "drop_rate", "window_goodput_rps"],
                                ["SLO attainment", "Drop rate", "Goodput (req/s)"]):
        unified_summary.pivot(index="group", columns="policy", values=metric).reindex(columns=order).plot.bar(ax=ax, rot=0)
        ax.set_title(title)
        if metric != "window_goodput_rps":
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.set_ylim(0, 1.05 if metric == "slo_attainment" else max(0.05, float(unified_summary[metric].max())*1.1))
    fig.tight_layout()
    fig.savefig(destination / "unified_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    fig, axes = plt.subplots(len(ranges), 1, figsize=(14, 4*len(ranges)), squeeze=False)
    for ax, group in zip(axes[:, 0], ranges):
        for policy in order:
            rows = unified_timeline[(unified_timeline["group"] == group["group"]) & (unified_timeline["policy"] == policy)].sort_values("source_start_s")
            ax.plot(rows["source_start_s"], rows["slo_attainment"].where(rows["arrivals"] > 0), label=policy)
        ax.set(title=group["group"], xlabel="Original CSV timestamp (seconds)", ylabel="Arrival-cohort SLO attainment", ylim=(0, 1.05))
        ax.ticklabel_format(style="plain", axis="x", useOffset=False)
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.legend()
    fig.tight_layout()
    fig.savefig(destination / "unified_timeline.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    columns = ["group", "policy", "arrivals", "slo_met", "slo_attainment", "drop_rate", "window_goodput_rps"]
    display(unified_summary[columns])
    parts = ["<!doctype html><html><head><meta charset='utf-8'><title>Unified policy comparison</title>",
             "<style>body{font:16px system-ui;max-width:1400px;margin:30px auto;padding:16px}img{max-width:100%}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px}pre{white-space:pre-wrap}</style></head><body>",
             "<h1>Unified continuous-replay policy comparison</h1><p>Same traffic, hardware, completion SLO, and intervals for all five policies. Existing DQN results reused; only the three rule-based policies were added. Online adaptation is not frozen held-out evaluation.</p>",
             unified_summary[columns].to_html(index=False),
             "<p>SLO denominator includes all arrivals, including drops. Timeline scores use eventual arrival-cohort outcomes for reporting only. Goodput counts on-time completions of each group's arrivals within its interval; gaps inside intervals contribute duration. Queues persist across bins/groups; omitted gaps are treated as idle.</p>"]
    for filename in ("unified_comparison.png", "unified_timeline.png"):
        parts.append("<img alt='" + filename + "' src='data:image/png;base64," + base64.b64encode((destination / filename).read_bytes()).decode() + "'>")
    parts += ["<h2>Actual replay settings</h2><pre>", html.escape(json.dumps(completed, indent=2)), "</pre></body></html>"]
    (destination / "unified_report.html").write_text("\n".join(parts), encoding="utf-8")
    members = [destination / f for f in ("unified_summary.csv", "unified_timeline.csv", "provenance.json",
               "unified_report.html", "unified_comparison.png", "unified_timeline.png")]
    members += [destination / p / f for p in order[:3] for f in ("summary.csv", "timeline.csv", "run.json")]
    archive = destination / "unified_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in members:
            bundle.write(path, path.relative_to(destination).as_posix())
    display(Image(filename=str(destination / "unified_comparison.png")))
    display(FileLink(str(destination / "unified_report.html")))
    display(FileLink(str(archive)))
    return unified_summary, unified_timeline, archive


# These paths were created by the existing notebook. Do not rerun its settings
# or training cells: that could point to a new, empty RUN_ROOT instead.
if globals().get("RUN_RULE_BASELINES", True):
    UNIFIED_DIR = Path(ONLINE_OUTPUT).parent / "unified_comparison"
    unified_summary, unified_timeline, UNIFIED_ARCHIVE = add_continuous_baselines(
        Path(ONLINE_OUTPUT), Path(TRAINING_DIR), Path(DATA_DIR), UNIFIED_DIR)
    if globals().get("IS_COLAB", False):
        from google.colab import files
        files.download(str(UNIFIED_ARCHIVE))
else:
    print("Rule-based comparison skipped; existing DQN report remains available.")
